# 04 - Model Selection

This notebook compares the logistic-regression baseline with XGBoost using stratified cross-validation on the training partition. Average precision (PR AUC) is the primary selection metric because churn is the minority class; ROC AUC, recall, precision, and F1 provide complementary context. The untouched test partition is evaluated only after selecting the candidate.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, make_scorer, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from churn_ml.models.evaluate_model import classification_metrics

RANDOM_STATE = 42
TARGET_COLUMN = "Churn Value"
CHURN_THRESHOLD = None  # Set a value from 0 to 1 to override the training churn-rate threshold.
def find_project_file(relative_path):
    for directory in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidate = directory / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {relative_path} from {Path.cwd()} or its parent directories.")

LOGISTIC_DATA_PATH = find_project_file(Path("data/processed/prediction_df_logistic_regression.csv"))
XGBOOST_DATA_PATH = find_project_file(Path("data/processed/prediction_df_xgboost.csv"))
VALUE_DATA_PATH = find_project_file(Path("data/raw/Telco_customer_churn.csv"))

logistic_model_df = pd.read_csv(LOGISTIC_DATA_PATH)
xgboost_model_df = pd.read_csv(XGBOOST_DATA_PATH)
value_df = pd.read_csv(VALUE_DATA_PATH, usecols=["CustomerID", TARGET_COLUMN, "CLTV"])
assert logistic_model_df[TARGET_COLUMN].equals(xgboost_model_df[TARGET_COLUMN])
if len(value_df) != len(logistic_model_df) or not value_df[TARGET_COLUMN].equals(logistic_model_df[TARGET_COLUMN]):
    raise ValueError("Raw CLTV values are not aligned with the processed modeling data.")

y = logistic_model_df[TARGET_COLUMN]
train_index, test_index = train_test_split(
    logistic_model_df.index, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
y_train, y_test = y.loc[train_index], y.loc[test_index]
ltv_test = value_df.loc[test_index, "CLTV"].rename("predicted_ltv_if_retained")
customer_id_test = value_df.loc[test_index, "CustomerID"]
X_train_by_model = {
    "logistic_regression": logistic_model_df.drop(columns=TARGET_COLUMN).loc[train_index],
    "xgboost": xgboost_model_df.drop(columns=TARGET_COLUMN).loc[train_index],
}
X_test_by_model = {
    "logistic_regression": logistic_model_df.drop(columns=TARGET_COLUMN).loc[test_index],
    "xgboost": xgboost_model_df.drop(columns=TARGET_COLUMN).loc[test_index],
}
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
decision_threshold = float(y_train.mean()) if CHURN_THRESHOLD is None else float(CHURN_THRESHOLD)
if not 0 < decision_threshold < 1:
    raise ValueError("CHURN_THRESHOLD must be between 0 and 1.")
print(f"Prediction threshold: {decision_threshold:.1%}")

Prediction threshold: 26.5%


## Candidate pipelines

The logistic pipeline is the interpretable benchmark. XGBoost is the nonlinear candidate. Imputation and scaling are inside the logistic pipeline so each cross-validation fold learns them only from its own training data.

In [2]:
candidate_models = {
    "logistic_regression": Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(max_iter=1_000, class_weight="balanced", random_state=RANDOM_STATE)),
        ]
    ),
    "xgboost": XGBClassifier(
        objective="binary:logistic", eval_metric="logloss", n_estimators=400,
        max_depth=4, learning_rate=0.05, subsample=0.80, colsample_bytree=0.80,
        scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE, n_jobs=1,
    ),
}
def thresholded_precision(y_true, y_proba):
    return precision_score(y_true, y_proba >= decision_threshold, zero_division=0)

def thresholded_recall(y_true, y_proba):
    return recall_score(y_true, y_proba >= decision_threshold, zero_division=0)

def thresholded_f1(y_true, y_proba):
    return f1_score(y_true, y_proba >= decision_threshold, zero_division=0)

scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "precision": make_scorer(thresholded_precision, response_method="predict_proba"),
    "recall": make_scorer(thresholded_recall, response_method="predict_proba"),
    "f1": make_scorer(thresholded_f1, response_method="predict_proba"),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## Cross-validated comparison and holdout evaluation

Select the highest mean cross-validated PR AUC. If performance is similar, prefer logistic regression for its simpler, more transparent behavior; record that decision explicitly.

In [3]:
cv_rows = []
for model_name, model in candidate_models.items():
    scores = cross_validate(model, X_train_by_model[model_name], y_train, scoring=scoring, cv=cv, n_jobs=-1)
    cv_rows.append({
        "model": model_name,
        **{f"mean_{metric}": scores[f"test_{metric}"].mean() for metric in scoring},
        **{f"std_{metric}": scores[f"test_{metric}"].std() for metric in scoring},
    })

cv_results = pd.DataFrame(cv_rows).sort_values("mean_pr_auc", ascending=False).set_index("model")
display(cv_results)

selected_model_name = cv_results.index[0]
holdout_rows = []
fitted_models = {}
test_probabilities = {}
for model_name, model in candidate_models.items():
    fitted_model = clone(model).fit(X_train_by_model[model_name], y_train)
    test_proba = fitted_model.predict_proba(X_test_by_model[model_name])[:, 1]
    test_pred = (test_proba >= decision_threshold).astype(int)
    holdout_rows.append({"model": model_name, **classification_metrics(y_test, test_pred, test_proba)})
    fitted_models[model_name] = fitted_model
    test_probabilities[model_name] = test_proba

holdout_metrics = pd.DataFrame(holdout_rows).set_index("model")
print(f"Selected model by mean CV PR AUC: {selected_model_name}")
display(holdout_metrics)

test_proba = test_probabilities[selected_model_name]
test_pred = (test_proba >= decision_threshold).astype(int)
confusion = confusion_matrix(y_test, test_pred, labels=[0, 1])
fig = go.Figure(go.Heatmap(
    z=confusion, x=["Predicted: no churn", "Predicted: churn"],
    y=["Actual: no churn", "Actual: churn"],
    colorscale="Blues", text=confusion, texttemplate="%{text}",
    colorbar={"title": "Customers"},
))
fig.update_layout(title=f"{selected_model_name} Confusion Matrix (threshold = {decision_threshold:.1%})")
fig.show()

,mean_pr_auc,mean_roc_auc,mean_precision,mean_recall,mean_f1,std_pr_auc,std_roc_auc,std_precision,std_recall,std_f1
model,,,,,,,,,,
xgboost,0.679195,0.856792,0.461625,0.903679,0.610996,0.016757,0.008420,0.004553,0.020442,0.005586
logistic_regression,0.678589,0.857506,0.429117,0.927759,0.586767,0.017557,0.012705,0.009408,0.017773,0.011167


Selected model by mean CV PR AUC: xgboost


,accuracy,precision,recall,f1,pr_auc,roc_auc
model,,,,,,
logistic_regression,0.647977,0.425245,0.927807,0.583193,0.639283,0.846438
xgboost,0.688432,0.455902,0.898396,0.604860,0.666765,0.847525


## Expected-value comparison for retention targeting

Model quality alone does not determine which customers should receive a retention offer. This comparison retrieves holdout-test `CLTV` from the raw data and treats it as predicted lifetime value if the customer is retained. CLTV is not used as a churn-model feature.

For each model, the top 100 customers are ranked by expected net value:

$$P(\text{churn}) \times 10\% \times \text{predicted LTV if retained} - \$20 - (40\% \times \$500)$$

The scenario assumes $20 outreach cost for every target, a $500 offer, and a 40% acceptance rate among targeted customers—including customers who would have stayed without intervention. The expected offer cost is therefore $200 per target. The 10% retention uplift represents the share of would-be churners saved by outreach. The uplift and acceptance rate are business assumptions, not estimates from either churn model.

In [4]:
OUTREACH_COST = 20
OFFER_COST = 500
OFFER_ACCEPTANCE_RATE = 0.40
RETENTION_UPLIFT = 0.10
TARGET_COUNT = 100

targeting_rows = []
target_ids_by_model = {}
for model_name, test_proba in test_probabilities.items():
    candidates = pd.DataFrame({
        "CustomerID": customer_id_test.to_numpy(),
        "predicted_churn_probability": test_proba,
        "predicted_ltv_if_retained": ltv_test.to_numpy(),
    })
    candidates["expected_value_before_cost"] = (
        candidates["predicted_churn_probability"]
        * RETENTION_UPLIFT
        * candidates["predicted_ltv_if_retained"]
    )
    candidates["expected_offer_cost"] = OFFER_COST * OFFER_ACCEPTANCE_RATE
    candidates["campaign_cost"] = OUTREACH_COST + candidates["expected_offer_cost"]
    candidates["expected_net_value"] = (
        candidates["expected_value_before_cost"] - candidates["campaign_cost"]
    )
    top_targets = candidates.nlargest(TARGET_COUNT, "expected_net_value")
    target_ids_by_model[model_name] = set(top_targets["CustomerID"])
    targeting_rows.append({
        "model": model_name,
        "customers_targeted": len(top_targets),
        "expected_value_before_cost": top_targets["expected_value_before_cost"].sum(),
        "outreach_cost": OUTREACH_COST * len(top_targets),
        "expected_offer_cost": top_targets["expected_offer_cost"].sum(),
        "campaign_cost": top_targets["campaign_cost"].sum(),
        "expected_net_value": top_targets["expected_net_value"].sum(),
    })

targeting_results = (
    pd.DataFrame(targeting_rows)
    .sort_values("expected_net_value", ascending=False)
    .set_index("model")
)
display(targeting_results.style.format({
    "expected_value_before_cost": "${:,.2f}",
    "outreach_cost": "${:,.2f}",
    "expected_offer_cost": "${:,.2f}",
    "campaign_cost": "${:,.2f}",
    "expected_net_value": "${:,.2f}",
}))

shared_targets = len(set.intersection(*target_ids_by_model.values()))
print(f"Shared customers in the two top-{TARGET_COUNT} target lists: {shared_targets}")

,customers_targeted,expected_value_before_cost,outreach_cost,expected_offer_cost,campaign_cost,expected_net_value
model,,,,,,
logistic_regression,100,"$45,555.11","$2,000.00","$20,000.00","$22,000.00","$23,555.11"
xgboost,100,"$45,390.09","$2,000.00","$20,000.00","$22,000.00","$23,390.09"


Shared customers in the two top-100 target lists: 83


## Selection insights

Compare the cross-validated PR AUC, holdout metrics, expected net value, and the overlap between the two target lists. When expected values are effectively tied and the models select largely the same customers, logistic regression is the preferred operational model: it is simpler to explain, audit, and maintain.

A materially higher expected net value, or a consistent holdout and cross-validation advantage, would justify choosing XGBoost despite its additional complexity. Before deploying either approach, validate the retention uplift and offer-acceptance assumptions with a randomized campaign or other credible treatment-control design.